# Valuing Actions by Estimating Probabilities (VAEP)

## Import Libraries

In [2]:
# Ensure that no .pyc files are generated
import sys

sys.dont_write_bytecode = True

In [ ]:
import pandas as pd
import socceraction.vaep.formula as vaep_formula
from socceraction import spadl
from tqdm import tqdm

from config import paths

## Load SPADL Data

In [4]:
with pd.HDFStore(paths.SPADL_H5) as spadl_store:
    games_df = (
        spadl_store["games"]
        .merge(spadl_store["competitions"], how="left")
        .merge(spadl_store["teams"].add_prefix("home_"), how="left")
        .merge(spadl_store["teams"].add_prefix("away_"), how="left")
    )
    players_df = spadl_store["players"]
    teams_df = spadl_store["teams"]

## Compute VAEP values

In [ ]:
actions_value = []

for game in tqdm(list(games_df.itertuples()), desc="Rating actions"):
    actions_df = pd.read_hdf(paths.SPADL_H5, f"actions/game_{game.game_id}")
    actions_df = (
        spadl.add_names(actions_df)  # type: ignore
        .merge(players_df, how="left")
        .merge(teams_df, how="left")
        .sort_values(["game_id", "period_id", "action_id"])
        .reset_index(drop=True)
    )
    preds = pd.read_hdf(paths.PREDICTIONS_H5, f"game_{game.game_id}")
    values = vaep_formula.value(actions_df, preds.scores, preds.concedes)  # type: ignore
    actions_value.append(pd.concat([actions_df, preds, values], axis=1))

actions_value = pd.concat(actions_value).sort_values(["game_id", "period_id", "time_seconds"]).reset_index(drop=True)

Rating actions: 100%|██████████| 2085/2085 [02:21<00:00, 14.71it/s]


## Analyze VAEP Rating

In [ ]:
actions_value["count"] = 1  # type: ignore

# Compute each player's number of actions and total VAEP values
players_vaep_df = (
    actions_value[["player_id", "vaep_value", "offensive_value", "defensive_value", "count"]]  # type: ignore
    .groupby(["player_id"])
    .sum()
    .reset_index()
)

In [7]:
# Add player names
players_vaep_df = players_vaep_df.merge(players_df[["player_id", "nickname", "player_name"]], how="left")
players_vaep_df["player_name"] = players_vaep_df[["nickname", "player_name"]].apply(
    lambda x: x.iloc[0] if x.iloc[0] else x.iloc[1], axis=1
)

In [8]:
# Show results
players_vaep_df = players_vaep_df[
    ["player_id", "player_name", "vaep_value", "offensive_value", "defensive_value", "count"]
]
players_vaep_df.sort_values("vaep_value", ascending=False)[:10]

,player_id,player_name,vaep_value,offensive_value,defensive_value,count
864,5503.0,Lionel Messi,32.100432,36.756817,-4.656385,6303
454,3672.0,Zlatan Ibrahimović,23.248282,24.274554,-1.026272,2914
792,5246.0,Luis Suárez,20.521909,21.430891,-0.908982,2973
2236,10955.0,Harry Kane,19.915256,20.920735,-1.005479,3788
848,5487.0,Antoine Griezmann,18.878598,19.187634,-0.309036,5019
493,3814.0,Riyad Mahrez,18.008978,18.384992,-0.376014,3065
572,4320.0,Neymar,17.681191,19.491947,-1.810756,5920
37,2995.0,Ángel Di María,17.575154,19.100584,-1.525430,4479
2628,19677.0,Karim Benzema,15.992427,16.402242,-0.409815,1902
935,5574.0,Toni Kroos,15.974898,15.898820,0.076077,7258


## Normalize per 90 minutes

In [9]:
# Normalize for minutes played
player_games_df = pd.read_hdf(paths.SPADL_H5, "player_games")
player_games_df = player_games_df[player_games_df.game_id.isin(games_df.game_id)]
player_minutes_df = player_games_df[["player_id", "minutes_played"]].groupby("player_id").sum().reset_index()

In [12]:
stats = players_vaep_df.merge(player_minutes_df)

stats = stats[stats.minutes_played > 180]  # At least two full games played

stats["vaep_rating"] = stats.vaep_value * 90 / stats.minutes_played
stats["offensive_rating"] = stats.offensive_value * 90 / stats.minutes_played
stats["defensive_rating"] = stats.defensive_value * 90 / stats.minutes_played

stats.sort_values("vaep_rating", ascending=False)[:10]

,player_id,player_name,vaep_value,offensive_value,defensive_value,count,minutes_played,vaep_rating,offensive_rating,defensive_rating
2532,16527.0,Mislav Oršić,2.690905,2.675378,0.015528,175,255,0.949731,0.944251,0.005480
3446,38803.0,Gonçalo Ramos,2.083995,2.111472,-0.027477,88,205,0.914925,0.926988,-0.012063
454,3672.0,Zlatan Ibrahimović,23.248282,24.274554,-1.026272,2914,2533,0.826034,0.862499,-0.036464
3111,28032.0,Ivan Schranz,3.015456,2.840406,0.175049,193,349,0.777625,0.732483,0.045142
3707,51733.0,Folarin Balogun,1.529309,1.528680,0.000629,82,181,0.760430,0.760117,0.000313
1845,8361.0,Ritsu Doan,1.917063,2.006498,-0.089434,150,235,0.734194,0.768446,-0.034251
834,5473.0,Ahmed Musa,1.812586,1.855238,-0.042652,161,224,0.728271,0.745408,-0.017137
528,3990.0,Jesé,7.054093,7.181148,-0.127055,951,897,0.707769,0.720517,-0.012748
2454,15780.0,Francesco Totti,3.642750,4.875775,-1.233025,629,472,0.694592,0.929703,-0.235111
3464,39624.0,Francisco Conceição,1.515183,1.581597,-0.066414,291,201,0.678440,0.708178,-0.029738
